# Description

It shows the pathways enriched (from the MultiPLIER models) given an LV name (in Settings below).
These "pathways enriched" are a set of limited pathways used during training of this PLIER model.
More pathways might be enriched if using external and more comprehensive databases such as gProfiler or FUMA as shown below.

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import re
from pathlib import Path

import pandas as pd

from entity import Trait
import conf

# Settings

In [3]:
LV_NAME = "LV24"

# Paths

In [4]:
OUTPUT_FIGURES_DIR = Path(conf.RESULTS_DIR, "demo", f"{LV_NAME.lower()}").resolve()
display(OUTPUT_FIGURES_DIR)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PosixPath('/opt/data/results/demo/lv24')

# Load MultiPLIER summary

In [5]:
multiplier_model_summary = pd.read_pickle(conf.MULTIPLIER["MODEL_SUMMARY_FILE"])

In [6]:
multiplier_model_summary.shape

(2157, 5)

In [7]:
multiplier_model_summary.head()

,pathway,LV index,AUC,p-value,FDR
1,KEGG_LYSINE_DEGRADATION,1,0.388059,0.866078,0.956005
2,REACTOME_MRNA_SPLICING,1,0.733057,0.000048,0.000582
3,MIPS_NOP56P_ASSOCIATED_PRE_RRNA_COMPLEX,1,0.680555,0.001628,0.011366
4,KEGG_DNA_REPLICATION,1,0.549473,0.312155,0.539951
5,PID_MYC_ACTIVPATHWAY,1,0.639303,0.021702,0.083739


# LV pathways

In [8]:
lv_pathways = multiplier_model_summary[
    multiplier_model_summary["LV index"].isin((LV_NAME[2:],))
    & (
        (multiplier_model_summary["FDR"] < 0.05)
                | (multiplier_model_summary["AUC"] >= 0.75)
    )
]

In [9]:
lv_pathways.shape

(1, 5)

In [10]:
lv_pathways = lv_pathways[["pathway", "AUC", "FDR"]].sort_values("FDR")

In [11]:
lv_pathways = lv_pathways.assign(AUC=lv_pathways["AUC"].apply(lambda x: f"{x:.2f}"))

In [12]:
lv_pathways = lv_pathways.assign(FDR=lv_pathways["FDR"].apply(lambda x: f"{x:.2e}"))

In [13]:
lv_pathways = lv_pathways.rename(
    columns={
        "pathway": "Pathway",
    }
)

In [14]:
lv_pathways.head()

,Pathway,AUC,FDR
130,PID_DELTANP63PATHWAY,0.78,9.06e-03


# Load LV data

In [15]:
from data.recount2 import LVAnalysis

In [16]:
lv_obj = LVAnalysis(LV_NAME)

Here I show the top 20 genes for our LV. You can see gene symbols, the LV weight (in column `LV603`) and the cytoband.

In [17]:
lv_obj.lv_genes.head(20)

,gene_name,LV24,gene_band
0,KCNK7,5.501678,11q13.1
1,KRT1,5.084971,12q13.13
2,ACER1,4.931350,19p13.3
3,ASPRV1,4.901245,2p13.3
4,CDHR1,4.253085,10q23.1
5,CTNNBIP1,3.859810,1p36.22
6,CAPNS2,3.379442,16q12.2
7,ELOVL3,3.331303,10q24.32
8,KLC3,3.326713,19q13.32
9,DSP,3.114874,6p24.3


# Pathway enrichment using external databases

## gProfiler

In [24]:
print(" ".join(lv_obj.lv_genes.head(70)["gene_name"].tolist()))

KCNK7 KRT1 ACER1 ASPRV1 CDHR1 CTNNBIP1 CAPNS2 ELOVL3 KLC3 DSP EVPL PERP JUP KRT5 WNT4 SULT2B1 EFNA3 EPHB6 DSC1 TMEM45A CD207 TGM5 ALDH3B2 EEF2K COL17A1 GJB5 PPP1R13L AQP3 RARG DEGS1 SPTBN2 PARD6G SLC46A2 GSN ALOX15B FADS2 KREMEN1 MAP3K6 TP63 NOTCH3 EXOSC7 BARX2 SFN TNFRSF19 SLC6A9 SFRP2 BLMH RORA MAPK13 KLF4 KLF5 RXRA ALOX12B TNXB IMPA2 KRT23 EPHB3 DGAT2 CMA1 WNT16 CLTB ZNF385A PTPRF GPNMB ACVR1B CEBPA CRY2 TACSTD2 PAK6 ISM1


Copy/paste the list of genes above and use gProfiler: https://biit.cs.ut.ee/gprofiler/gost

Results URL: https://biit.cs.ut.ee/gplink/l/a5kY96GBjSk

**Notes**: seems to be very skin related.

## FUMA

In [26]:
# print top genes in module
print("\n".join(lv_obj.lv_genes.head(70)["gene_name"].tolist()))

KCNK7
KRT1
ACER1
ASPRV1
CDHR1
CTNNBIP1
CAPNS2
ELOVL3
KLC3
DSP
EVPL
PERP
JUP
KRT5
WNT4
SULT2B1
EFNA3
EPHB6
DSC1
TMEM45A
CD207
TGM5
ALDH3B2
EEF2K
COL17A1
GJB5
PPP1R13L
AQP3
RARG
DEGS1
SPTBN2
PARD6G
SLC46A2
GSN
ALOX15B
FADS2
KREMEN1
MAP3K6
TP63
NOTCH3
EXOSC7
BARX2
SFN
TNFRSF19
SLC6A9
SFRP2
BLMH
RORA
MAPK13
KLF4
KLF5
RXRA
ALOX12B
TNXB
IMPA2
KRT23
EPHB3
DGAT2
CMA1
WNT16
CLTB
ZNF385A
PTPRF
GPNMB
ACVR1B
CEBPA
CRY2
TACSTD2
PAK6
ISM1


In [27]:
# save all genes in model to use as background list of genes
lv_obj.lv_genes["gene_name"].to_csv(OUTPUT_FIGURES_DIR / "all_genes.txt", header=None, index=False)

In [28]:
OUTPUT_FIGURES_DIR / "all_genes.txt"

PosixPath('/opt/data/results/demo/lv24/all_genes.txt')

In [29]:
!head /opt/data/results/demo/lv24/all_genes.txt

KCNK7
KRT1
ACER1
ASPRV1
CDHR1
CTNNBIP1
CAPNS2
ELOVL3
KLC3
DSP


In [30]:
!wc -l /opt/data/results/demo/lv24/all_genes.txt

6750 /opt/data/results/demo/lv24/all_genes.txt


Now go to the FUMA GENE2FUNC module here: https://fuma.ctglab.nl/gene2func

1. Paste the list of top genes above and then upload the `all_genes.txt` file.
2. Use a "Title" and click on "Submit"